In [ ]:
# Install pydub
!pip install pydub

# Ensure ffmpeg is available (only necessary on some platforms like Google Colab)
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install accelerate -U

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
  

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import torch
import torchaudio
import os
import numpy as np
from datasets import Dataset, load_metric
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import random
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score

def load_audio_files(audio_path, target_sr=16000, max_duration=25):
    audio_data = []
    labels = []

    singer_folders = os.listdir(audio_path)

    for singer in singer_folders:
        singer_path = os.path.join(audio_path, singer)
        audio_files = os.listdir(singer_path)

        for audio_file in audio_files[:20]:  # Increased from 15 to 20 files per singer
            file_path = os.path.join(singer_path, audio_file)

            # Load and resample audio using torchaudio
            waveform, sample_rate = torchaudio.load(file_path)
            if sample_rate != target_sr:
                waveform = torchaudio.functional.resample(waveform, sample_rate, target_sr)

            # Convert to mono if stereo
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Trim or pad to max_duration
            if waveform.shape[1] > max_duration * target_sr:
                waveform = waveform[:, :max_duration * target_sr]
            else:
                waveform = torch.nn.functional.pad(waveform, (0, max_duration * target_sr - waveform.shape[1]))

            # Normalize
            waveform = waveform / torch.max(torch.abs(waveform))

            audio_data.append(waveform.squeeze().numpy())
            labels.append(singer)

    return np.array(audio_data, dtype=object), np.array(labels)

# Usage
audio_path = '/content/drive/MyDrive/DATASET/'
audio_data, labels = load_audio_files(audio_path)

unique_labels = np.unique(labels)
print("Unique labels (singer names):", unique_labels)

# Create a mapping of labels to integers
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

# Convert string labels to integer ids
label_ids = np.array([label_to_id[label] for label in labels])

# Split the data into training and validation sets
train_data, val_data, train_labels, val_labels = train_test_split(
    audio_data, label_ids, test_size=0.2, random_state=42
)

# Create datasets
train_dataset = Dataset.from_dict({"input_values": train_data.tolist(), "label": train_labels.tolist()})
val_dataset = Dataset.from_dict({"input_values": val_data.tolist(), "label": val_labels.tolist()})

# Initialize feature extractor and model for audio classification
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
classification_model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=len(unique_labels),
    ignore_mismatched_sizes=True
)

def data_collator(features):
    input_features = [feature["input_values"] for feature in features]
    label_ids = [feature["label"] for feature in features]

    inputs = feature_extractor(
        input_features,
        sampling_rate=16000,
        padding="max_length",
        max_length=int(16000 * 30),  # 30 seconds max length
        truncation=True,
        return_tensors="pt"
    )

    inputs["labels"] = torch.tensor(label_ids)
    return inputs

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    metric = load_metric("accuracy")
    return metric.compute(predictions=predictions, references=labels)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=40,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",  # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    save_steps=1000,
    eval_steps=1000,
    dataloader_num_workers=2,  # Reduced from 4 to 2
    max_grad_norm=0.5,
    gradient_checkpointing=True,  # Added this line
)

# Create Trainer
trainer = Trainer(
    model=classification_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

def test_multiple_samples(model, feature_extractor, audio_path, id_to_label, num_samples=20):
    predictions = []
    true_labels = []

    for singer in os.listdir(audio_path):
        singer_path = os.path.join(audio_path, singer)
        audio_files = os.listdir(singer_path)

        for _ in range(num_samples // len(os.listdir(audio_path))):
            audio_file = random.choice(audio_files)
            file_path = os.path.join(singer_path, audio_file)

            waveform, sample_rate = torchaudio.load(file_path)
            if sample_rate != 16000:
                waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            waveform = waveform.squeeze().numpy()

            inputs = feature_extractor(waveform, sampling_rate=16000, return_tensors="pt", padding="max_length", max_length=int(16000 * 30), truncation=True)
            inputs = {k: v.to('cuda').half() for k, v in inputs.items()}

            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_class_id = logits.argmax().item()
            predicted_label = id_to_label[predicted_class_id]

            predictions.append(predicted_label)
            true_labels.append(singer)

    accuracy = accuracy_score(true_labels, predictions)
    return predictions, true_labels, accuracy

# Clear CUDA cache
torch.cuda.empty_cache()

# Save the trained model
trainer.save_model("./trained_audio_classifier")

# Move model to GPU and convert to half precision
classification_model = classification_model.to('cuda').half()

# Run multi-sample test
predictions, true_labels, accuracy = test_multiple_samples(classification_model, feature_extractor, audio_path, id_to_label)

print("Prediction distribution:", Counter(predictions))
print("True label distribution:", Counter(true_labels))
print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(true_labels, predictions))

Unique labels (singer names): ['SG' 'SPB' 'SRM']


/usr/local/lib/python3.10/dist-packages/transformers/configuration_utils.py:364: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multit

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.093180,0.500000
2,No log,1.093099,0.500000
3,No log,1.093099,0.500000
4,No log,1.092855,0.500000
5,No log,1.092855,0.500000
6,No log,1.092611,0.500000
7,No log,1.092611,0.500000
8,No log,1.091878,0.500000
9,No log,1.091553,0.500000
10,No log,1.090576,0.500000


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWa

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.093180,0.500000
2,No log,1.093099,0.500000
3,No log,1.093099,0.500000
4,No log,1.092855,0.500000
5,No log,1.092855,0.500000
6,No log,1.092611,0.500000
7,No log,1.092611,0.500000
8,No log,1.091878,0.500000
9,No log,1.091553,0.500000
10,No log,1.090576,0.500000


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork

Prediction distribution: Counter({'SRM': 12, 'SG': 6})
True label distribution: Counter({'SPB': 6, 'SRM': 6, 'SG': 6})
Accuracy: 0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

          SG       1.00      1.00      1.00         6
         SPB       0.00      0.00      0.00         6
         SRM       0.50      1.00      0.67         6

    accuracy                           0.67        18
   macro avg       0.50      0.67      0.56        18
weighted avg       0.50      0.67      0.56        18



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# Test the model on a single sample
test_audio_path = os.path.join(audio_path, unique_labels[0], os.listdir(os.path.join(audio_path, unique_labels[0]))[0])
waveform, sample_rate = torchaudio.load(test_audio_path)
if sample_rate != 16000:
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)
waveform = waveform.squeeze().numpy()

test_input = feature_extractor(waveform, sampling_rate=16000, return_tensors="pt", padding="max_length", max_length=int(16000 * 30), truncation=True)
test_input = {k: v.to('cuda').half() for k, v in test_input.items()}  # Move input to GPU and convert to half precision

with torch.no_grad():
    logits = classification_model(**test_input).logits
predicted_class_id = logits.argmax().item()
predicted_label = id_to_label[predicted_class_id]
print(f"\nPredicted singer for single sample: {predicted_label}")

In [ ]:
import torch
import torchaudio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# Load your trained model and feature extractor
model_name = "path/to/your/saved/model"  # trained_audio_classifier
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name)

# Function to preprocess and predict
def predict_singer(audio_file_path):
    # Load the audio file
    waveform, sample_rate = torchaudio.load(audio_file_path)

    # Resample if necessary (assuming 16kHz is required)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(sample_rate, 16000)
        waveform = resampler(waveform)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Preprocess the audio
    inputs = feature_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")

    # Make prediction
    with torch.no_grad():
        logits = model(**inputs).logits

    # Get the predicted class
    predicted_class_id = logits.argmax().item()

    # Map the class ID to singer name (you'll need to create this mapping)
    singer_names = ["Singer1", "Singer2", "Singer3"]  # Replace with your actual singer names
    predicted_singer = singer_names[predicted_class_id]

    return predicted_singer

# Use the function
audio_file_path = "path/to/your/audio/file.wav"
predicted_singer = predict_singer(audio_file_path)
print(f"The predicted singer is: {predicted_singer}")